# 04 - DRC HBR and Bucket Capital

Synthetic engineering and validation evidence, not final regulatory capital.

Verify hedge benefit ratio and bucket capital mechanics after showing the public capital entrypoint.

Use this notebook as a companion to [`PACKAGE_JOURNEY.md`](../docs/PACKAGE_JOURNEY.md) and [`examples/run_demo.py`](../examples/run_demo.py). The happy path is the public package API: construct DRC positions and a calculation context, then call `frtb_drc.calculate_drc_capital`.

## Raw inputs your upstream risk engine must emit

A DRC upstream engine should emit one row per default-risk position with stable position and source-row identifiers, desk/legal-entity scope, risk class, instrument type, issuer or tranche/index identifiers where applicable, bucket key, seniority or credit quality, notional or market value, maturity, currency, lineage fields, and citation identifiers. The non-securitisation table contract is documented in [`docs/schemas/input_table/frtb_drc.nonsec.schema.json`](../../../docs/schemas/input_table/frtb_drc.nonsec.schema.json); package-specific context and fixture notes live in [`DATASET_CONTRACT.md`](../docs/DATASET_CONTRACT.md).


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
for path in (ROOT, ROOT / "src"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from examples.notebook_utils import (
    DRC_BUCKET_COLORS,
    DRC_SERIES_COLORS,
    setup_notebook_plot_style,
)

from examples.drc_nonsec_fixture import load_drc_nonsec_v2_fixture, run_fixture_workflow
from frtb_drc.demo_data import ALL_POSITIONS, DEMO_CONTEXT

setup_notebook_plot_style()


def markdown_table(headers: list[str], rows: list[list[str]]) -> Markdown:
    header = "| " + " | ".join(headers) + " |"
    separator = "| " + " | ".join(["---"] * len(headers)) + " |"
    body = ["| " + " | ".join(row) + " |" for row in rows]
    return Markdown("\n".join([header, separator, *body]))


fixture = load_drc_nonsec_v2_fixture()
positions = fixture.positions
context = fixture.context
expected = fixture.expected_outputs
display(
    Markdown(
        f"Loaded fixture `{expected['fixture_id']}`: "
        f"`{len(positions)}` positions, "
        f"profile `{context.profile_id}`, "
        f"as-of `{context.calculation_date}`."
    )
)


## Public API happy path

The primary user contract is public API first: pass validated `DrcPosition` records and `DrcCalculationContext` to `calculate_drc_capital`. The implementation-detail sections below inspect intermediate mechanics for validation and auditability.


## Bucket capital flow

```mermaid
flowchart LR
    A[Net long JTD] --> C[Risk-weighted long]
    B[Net short JTD] --> D[Risk-weighted short]
    C --> E[HBR ratio]
    D --> E
    E --> F[Bucket capital]
    F --> G[Category total]
```


In [ ]:
from frtb_drc import calculate_drc_capital

public_result = calculate_drc_capital(positions, context=context)
print(f"Total DRC capital: {public_result.total_drc:,.2f}")
print(f"Input hash       : {public_result.input_hash}")
print(f"Profile hash     : {public_result.profile_hash}")


## Implementation details / math verification - Bucket capital mechanics


In [ ]:
from frtb_drc import calculate_drc_capital

result = calculate_drc_capital(positions, context=context)
cat = result.categories[0]

rows = [
    [
        b.bucket_key,
        f"{b.hbr.aggregate_net_long:>14,.2f}",
        f"{b.hbr.aggregate_net_short:>14,.2f}",
        f"{b.hbr.ratio:.6f}",
        f"{b.weighted_long:>14,.2f}",
        f"{b.weighted_short:>14,.2f}",
        f"{b.capital:>14,.2f}",
        "Y" if b.floor_applied else "",
    ]
    for b in cat.bucket_results
]
display(markdown_table(
    ["Bucket", "Agg net long", "Agg net short", "HBR", "W-long", "W-short", "Capital", "Floor?"],
    rows,
))


## HBR mechanics

### CORPORATE — partial hedge

`eta-finance` and `zeta-metals` have REJECTED shorts that survive as net short
positions.  After risk-weighting:

- `weighted_short > 0` → HBR < 1.0
- Capital = weighted\_long - HBR * weighted\_short

The HBR ensures the hedge benefit is shared proportionally across longs and shorts.

In [ ]:
corp = next(b for b in cat.bucket_results if b.bucket_key == "CORPORATE")
print(f"CORPORATE")
print(f"  aggregate_net_long  = {corp.hbr.aggregate_net_long:,.2f}")
print(f"  aggregate_net_short = {corp.hbr.aggregate_net_short:,.2f}")
print(f"  HBR                 = {corp.hbr.ratio:.6f}")
print(f"  weighted_long       = {corp.weighted_long:,.2f}")
print(f"  weighted_short      = {corp.weighted_short:,.2f}")
print(f"  capital (unfloored) = {corp.weighted_long - corp.hbr.ratio * corp.weighted_short:,.2f}")
print(f"  capital (final)     = {corp.capital:,.2f}")


### NON\_US\_SOVEREIGN — no net shorts after netting

UK, Japan, and Brazil shorts were fully consumed by their respective longs during netting.
No net short positions remain.  HBR = 1.0, but `weighted_short = 0` — so HBR has
no practical effect.

In [ ]:
sov = next(b for b in cat.bucket_results if b.bucket_key == "NON_US_SOVEREIGN")
print(f"NON_US_SOVEREIGN")
print(f"  aggregate_net_long  = {sov.hbr.aggregate_net_long:,.2f}")
print(f"  aggregate_net_short = {sov.hbr.aggregate_net_short:,.2f}")
print(f"  HBR                 = {sov.hbr.ratio:.6f}")
print(f"  capital             = {sov.capital:,.2f}")


## Bucket capital waterfall

Figure: DRC bucket-capital waterfall comparing risk-weighted longs, HBR-adjusted shorts, and resulting bucket capital.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

labels = [b.bucket_key for b in cat.bucket_results]
w_longs = [b.weighted_long / 1e6 for b in cat.bucket_results]
w_shorts_reduced = [b.hbr.ratio * b.weighted_short / 1e6 for b in cat.bucket_results]
capitals = [b.capital / 1e6 for b in cat.bucket_results]

x = np.arange(len(labels))
width = 0.28

fig, ax = plt.subplots(figsize=(9, 4.5))
bars1 = ax.bar(x - width, w_longs, width, label="RW * net long", color=DRC_SERIES_COLORS["long"], alpha=0.85)
bars2 = ax.bar(x, w_shorts_reduced, width, label="HBR * RW * net short", color=DRC_SERIES_COLORS["short"], alpha=0.85)
bars3 = ax.bar(x + width, capitals, width, label="Bucket capital", color=DRC_SERIES_COLORS["capital"], alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=15, ha="right", fontsize=9)
ax.set_ylabel("USD M")
ax.set_title("Bucket capital waterfall: RW * long - HBR * RW * short")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()


## See also

- [`packages/frtb-drc/docs/PACKAGE_JOURNEY.md`](../docs/PACKAGE_JOURNEY.md) for the package workflow and maturity path.
- [`packages/frtb-drc/docs/DATASET_CONTRACT.md`](../docs/DATASET_CONTRACT.md) for fixture and input-shape notes.
- [`docs/CLIENT_INTEGRATION.md`](../../../docs/CLIENT_INTEGRATION.md) for suite-level client integration guidance.
- Sibling component journeys: [`frtb-sbm`](../../frtb-sbm/docs/PACKAGE_JOURNEY.md), [`frtb-rrao`](../../frtb-rrao/docs/PACKAGE_JOURNEY.md), [`frtb-cva`](../../frtb-cva/docs/PACKAGE_JOURNEY.md), [`frtb-ima`](../../frtb-ima/docs/PACKAGE_JOURNEY.md), and [`frtb-orchestration`](../../frtb-orchestration/docs/PACKAGE_JOURNEY.md).
